In [ ]:
!npm install -g localtunnel


In [ ]:
import os
os.chdir('/content')
!rm -rf /content/venv
!pip install virtualenv
!virtualenv /content/venv
!rm -rf /content/usman-and-umar-app
!git clone --single-branch --branch feature/hybrid-task-orchestrator https://github.com/saedhawwa23-rgb/usman-and-umar-app.git /content/usman-and-umar-app
!/content/venv/bin/pip install -r /content/usman-and-umar-app/cloud-backend/requirements.txt
!/content/venv/bin/pip install -r /content/usman-and-umar-app/cloud-backend/requirements-ai.txt
!/content/venv/bin/python -m playwright install chromium
from google.colab import auth, userdata
auth.authenticate_user()
for secret_name in ('APP_CLIENT_TOKEN', 'FIRESTORE_PROJECT_ID', 'DRIVE_CHECKPOINT_FOLDER_ID', 'OPENSWARM_SERVER_ID', 'NVIDIA_API_KEY', 'OPENROUTER_API_KEY', 'GITHUB_TOKEN'):
    try:
        value = userdata.get(secret_name)
        if value:
            os.environ[secret_name] = value
    except Exception:
        pass
required = ('APP_CLIENT_TOKEN', 'FIRESTORE_PROJECT_ID', 'DRIVE_CHECKPOINT_FOLDER_ID', 'OPENSWARM_SERVER_ID')
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError('Missing required Colab Secrets: ' + ', '.join(missing))
print('Dependencies installed and Google authentication completed; secrets were not displayed.')


In [ ]:
import os, subprocess, time, requests
backend = '/content/usman-and-umar-app/cloud-backend'
os.chdir(backend)
log = open('/tmp/fastapi.log', 'w')
process = subprocess.Popen(['/content/venv/bin/python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'], stdout=log, stderr=subprocess.STDOUT)
ready = False
last_error = 'health endpoint did not respond'
for _ in range(60):
    if process.poll() is not None:
        break
    try:
        response = requests.get('http://127.0.0.1:7860/health', timeout=3)
        body = response.json()
        if response.status_code == 200 and body.get('status') == 'ready' and body.get('accepting_new_tasks') is True:
            ready = True
            print('Backend readiness checks passed:', {k: body.get(k) for k in ('server_id', 'instance_id', 'drive_ready', 'checkpoint_storage_ready', 'lease_coordinator_ready')})
            break
        last_error = str({k: body.get(k) for k in ('status', 'drive_ready', 'lease_coordinator_ready')})
    except Exception as exc:
        last_error = type(exc).__name__
    time.sleep(2)
if not ready:
    process.terminate()
    raise RuntimeError('Backend did not become ready; last health result: ' + last_error + '. Inspect /tmp/fastapi.log. No tunnel should be published.')


### Configure a per-server Localtunnel subdomain

In Google Colab, open **Secrets** (key icon), add a secret named `TUNNEL_SUBDOMAIN`, enter a unique value such as `usman-umar-1` for Server 1 or `usman-umar-2` for Server 2, and enable notebook access. If unset, the tunnel cell defaults to `usman-umar-1`.


In [ ]:
import os, time
try:
    from google.colab import userdata
    os.environ["TUNNEL_SUBDOMAIN"] = userdata.get("TUNNEL_SUBDOMAIN")
except Exception:
    pass  # Allows the environment-variable/default fallback outside Colab or when the secret is unavailable.
subdomain = os.environ.get("TUNNEL_SUBDOMAIN", "usman-umar-1")
!lt --port 7860 --subdomain {subdomain} > /tmp/localtunnel.log 2>&1 &
time.sleep(3)
print(f"Public URL: https://{subdomain}.loca.lt")
print("Open the URL in Chrome first to bypass the Localtunnel warning page.")


In [ ]:
# AI and browser dependencies were installed synchronously before the backend was started.
print('AI and browser dependencies were installed before readiness checks.')


In [ ]:
# Readiness is checked in the startup cell; this cell only confirms configured dependency imports.
import importlib.util
required_modules = ('langgraph', 'langchain_openai', 'playwright', 'google.cloud.firestore', 'googleapiclient')
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError('Required modules are unavailable: ' + ', '.join(missing_modules))
print('Configured backend dependency imports are available.')
